# Bases de Datos — Consultas y Cálculos

Notebook de acceso directo a todos los sistemas de almacenamiento del pipeline:
**InfluxDB · MinIO · Redpanda/Kafka · FastAPI**

| Sección | Base de datos | Qué hacemos |
|---------|--------------|-------------|
| 0 | — | Conexiones e imports |
| 1 | InfluxDB | Estadísticas hot path, alertas, correlaciones |
| 2 | Redpanda | Watermarks por topic, DLQ, ratio válidos/inválidos |
| 3 | MinIO | Inventario de archivos, DuckDB cold path, particiones |
| 4 | FastAPI | Estado de máquinas, endpoints, predicciones IA |
| 5 | Lambda Query | UNION ALL hot+cold, solapamiento, cobertura temporal |
| 6 | Cálculos | Z-score, tendencia, heatmap de correlación, ranking de anomalías |

---
## 0 — Setup y conexiones

In [ ]:
import os, json, warnings
import subprocess
import urllib.request
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from influxdb_client import InfluxDBClient
from minio import Minio
from confluent_kafka import Consumer, KafkaError, TopicPartition
from confluent_kafka.admin import AdminClient

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Configuración de conexión ──────────────────────────────────────────────
INFLUX_URL    = os.getenv('INFLUX_URL',   'http://influxdb:8086')
INFLUX_TOKEN  = os.getenv('INFLUX_TOKEN', 'supersecrettoken')
INFLUX_ORG    = os.getenv('INFLUX_ORG',   'ilerna')
INFLUX_BUCKET = os.getenv('INFLUX_BUCKET','sensores')

KAFKA_BROKER  = os.getenv('KAFKA_BROKER', 'redpanda:29092')

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'minio:9000')
MINIO_KEY      = os.getenv('MINIO_ACCESS_KEY', 'admin')
MINIO_SECRET   = os.getenv('MINIO_SECRET_KEY', 'Ilerna_Programaci0n')
MINIO_BUCKET   = os.getenv('MINIO_BUCKET', 'datalake')

API_URL = os.getenv('API_URL', 'http://localhost:8000')

print('Configuración cargada:')
print(f'  InfluxDB : {INFLUX_URL}')
print(f'  Kafka    : {KAFKA_BROKER}')
print(f'  MinIO    : {MINIO_ENDPOINT}')
print(f'  FastAPI  : {API_URL}')

In [ ]:
# ── Test de conectividad ───────────────────────────────────────────────────
results = {}

# InfluxDB
try:
    with InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG) as c:
        ok = c.ping()
    results['InfluxDB'] = '✅ OK'
except Exception as e:
    results['InfluxDB'] = f'❌ {e}'

# MinIO
try:
    mc = Minio(MINIO_ENDPOINT, access_key=MINIO_KEY, secret_key=MINIO_SECRET, secure=False)
    mc.bucket_exists(MINIO_BUCKET)
    results['MinIO'] = '✅ OK'
except Exception as e:
    results['MinIO'] = f'❌ {e}'

# Kafka / Redpanda
try:
    adm = AdminClient({'bootstrap.servers': KAFKA_BROKER})
    meta = adm.list_topics(timeout=5)
    results['Redpanda'] = f'✅ OK ({len(meta.topics)} topics)'
except Exception as e:
    results['Redpanda'] = f'❌ {e}'

# FastAPI
try:
    with urllib.request.urlopen(f'{API_URL}/health', timeout=5) as r:
        h = json.loads(r.read())
    results['FastAPI'] = f"✅ {h.get('status','?')} (influxdb={h.get('influxdb','?')})"
except Exception as e:
    results['FastAPI'] = f'❌ {e}'

for svc, status in results.items():
    print(f'  {svc:<12} {status}')

---
## 1 — InfluxDB: Hot Path

Consultas directas al measurement `machine_stats`. Cada fila representa una ventana Tumble de 1 minuto por máquina.

In [ ]:
# ── 1.1 Leer todos los datos de las últimas 3 horas ───────────────────────
FLUX_QUERY = '''
from(bucket: "{bucket}")
  |> range(start: -3h)
  |> filter(fn: (r) => r._measurement == "machine_stats")
  |> pivot(rowKey: ["_time", "device_id"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "device_id", "avg_temp_c", "max_temp_c", "count", "alert"])
'''.format(bucket=INFLUX_BUCKET)

with InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG) as client:
    df_hot = client.query_api().query_data_frame(FLUX_QUERY)

if df_hot.empty:
    print('⚠️  Sin datos en InfluxDB. Verifica que los jobs Flink estén corriendo.')
else:
    df_hot = df_hot.drop(columns=['result', 'table'], errors='ignore')
    df_hot['_time'] = pd.to_datetime(df_hot['_time'], utc=True)
    df_hot = df_hot.sort_values('_time').reset_index(drop=True)
    print(f'Filas cargadas : {len(df_hot)}')
    print(f'Ventanas       : {df_hot["_time"].nunique()} timestamps únicos')
    print(f'Máquinas       : {sorted(df_hot["device_id"].unique())}')
    print(f'Rango temporal : {df_hot["_time"].min()} → {df_hot["_time"].max()}')
    df_hot.head()

In [ ]:
# ── 1.2 Estadísticas por máquina ──────────────────────────────────────────
stats = df_hot.groupby('device_id').agg(
    ventanas       = ('avg_temp_c', 'count'),
    temp_media     = ('avg_temp_c', 'mean'),
    temp_max       = ('max_temp_c', 'max'),
    temp_min       = ('avg_temp_c', 'min'),
    desv_std       = ('avg_temp_c', 'std'),
    p25            = ('avg_temp_c', lambda x: x.quantile(0.25)),
    p75            = ('avg_temp_c', lambda x: x.quantile(0.75)),
    alertas        = ('alert', 'sum'),
    lecturas_total = ('count', 'sum'),
).round(2)

stats['pct_alerta'] = (stats['alertas'] / stats['ventanas'] * 100).round(1)
stats['iqr'] = (stats['p75'] - stats['p25']).round(2)
print('Estadísticas por máquina (últimas 3h):')
stats

In [ ]:
# ── 1.3 Gráfica: temperatura media por máquina ────────────────────────────
fig = go.Figure()
for device in sorted(df_hot['device_id'].unique()):
    d = df_hot[df_hot['device_id'] == device]
    fig.add_trace(go.Scatter(
        x=d['_time'], y=d['avg_temp_c'],
        mode='lines+markers', name=device,
        marker=dict(size=4)
    ))
fig.add_hline(y=80, line_dash='dash', line_color='red', annotation_text='Umbral 80°C')
fig.update_layout(
    title='Temperatura media por máquina — últimas 3h (InfluxDB)',
    xaxis_title='Tiempo', yaxis_title='°C',
    legend_title='Máquina', height=400
)
fig.show()

In [ ]:
# ── 1.4 Histograma de temperaturas por máquina ────────────────────────────
fig = px.histogram(
    df_hot, x='avg_temp_c', color='device_id',
    nbins=30, barmode='overlay', opacity=0.7,
    title='Distribución de temperatura media por máquina (InfluxDB)',
    labels={'avg_temp_c': 'Temperatura media (°C)', 'device_id': 'Máquina'}
)
fig.add_vline(x=80, line_dash='dash', line_color='red', annotation_text='80°C')
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── 1.5 Alertas: frecuencia y máquina más conflictiva ─────────────────────
alerts_df = df_hot[df_hot['alert'] == 1].copy()
print(f'Total ventanas en alerta: {len(alerts_df)}')
print(f'Total ventanas normales : {len(df_hot) - len(alerts_df)}')
print()

if not alerts_df.empty:
    alert_summary = alerts_df.groupby('device_id').agg(
        alertas      = ('alert', 'count'),
        temp_media   = ('avg_temp_c', 'mean'),
        temp_max_pico= ('max_temp_c', 'max'),
    ).sort_values('alertas', ascending=False)
    print('Máquinas con más alertas:')
    display(alert_summary)

    fig = px.bar(
        alert_summary.reset_index(), x='device_id', y='alertas',
        color='temp_media', color_continuous_scale='OrRd',
        title='Número de alertas por máquina (últimas 3h)',
        labels={'device_id': 'Máquina', 'alertas': 'Ventanas en alerta', 'temp_media': 'Temp media (°C)'}
    )
    fig.show()
else:
    print('Sin alertas en el período.')

In [ ]:
# ── 1.6 Consulta Flux directa: últimos valores (sin pivot) ────────────────
FLUX_LAST = '''
from(bucket: "{bucket}")
  |> range(start: -15m)
  |> filter(fn: (r) => r._measurement == "machine_stats" and r._field == "avg_temp_c")
  |> group(columns: ["device_id"])
  |> last()
'''.format(bucket=INFLUX_BUCKET)

with InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG) as client:
    df_last = client.query_api().query_data_frame(FLUX_LAST)

if not df_last.empty:
    df_last = df_last[['device_id', '_time', '_value']].rename(columns={'_value': 'avg_temp_c_actual'})
    df_last['estado'] = df_last['avg_temp_c_actual'].apply(lambda v: '🔴 ALERTA' if v > 80 else '🟢 OK')
    print('Estado actual de máquinas (último valor en los últimos 15 min):')
    display(df_last.sort_values('device_id'))
else:
    print('Sin datos recientes en InfluxDB.')

---
## 2 — Redpanda / Kafka: Topics y mensajes

In [ ]:
# ── 2.1 Watermark (total mensajes) por topic ──────────────────────────────
TOPICS = ['sensors_raw', 'sensors_clean', 'sensors_verified', 'sensors_invalid']

consumer = Consumer({
    'bootstrap.servers': KAFKA_BROKER,
    'group.id': 'notebook-analysis-03',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
})

watermarks = {}
for topic in TOPICS:
    try:
        tp = TopicPartition(topic, 0)
        lo, hi = consumer.get_watermark_offsets(tp, timeout=5)
        watermarks[topic] = {'low': lo, 'high': hi, 'mensajes': hi - lo}
    except Exception as e:
        watermarks[topic] = {'low': 0, 'high': 0, 'mensajes': 0, 'error': str(e)}

consumer.close()

df_wm = pd.DataFrame(watermarks).T
print('Mensajes por topic (high watermark):')
display(df_wm)

fig = px.bar(
    df_wm.reset_index().rename(columns={'index': 'topic'}),
    x='topic', y='mensajes',
    color='topic', text='mensajes',
    title='Total de mensajes por topic (Redpanda)',
    color_discrete_map={
        'sensors_raw': 'steelblue',
        'sensors_clean': 'green',
        'sensors_verified': 'teal',
        'sensors_invalid': 'orange',
    }
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=350)
fig.show()

In [ ]:
# ── 2.2 Ratio valid/invalid y tasa de rechazo ─────────────────────────────
raw_total     = watermarks.get('sensors_raw', {}).get('mensajes', 0)
clean_total   = watermarks.get('sensors_clean', {}).get('mensajes', 0)
invalid_total = watermarks.get('sensors_invalid', {}).get('mensajes', 0)
verified_total= watermarks.get('sensors_verified', {}).get('mensajes', 0)

print(f'sensors_raw      : {raw_total:>6} mensajes  (100%)')
if raw_total > 0:
    print(f'sensors_clean    : {clean_total:>6} mensajes  ({clean_total/raw_total*100:.1f}% de raw)')
    print(f'sensors_invalid  : {invalid_total:>6} mensajes  ({invalid_total/raw_total*100:.1f}% de raw)  ← DLQ')
    print(f'sensors_verified : {verified_total:>6} mensajes  ({verified_total/raw_total*100:.1f}% de raw)  ← hash OK')
    print()
    print(f'Tasa de aceptación  : {clean_total/(raw_total)*100:.1f}%')
    print(f'Tasa de rechazo DLQ : {invalid_total/(raw_total)*100:.1f}%')

# Pie chart
if raw_total > 0:
    labels = ['sensors_clean', 'sensors_invalid']
    values = [clean_total, invalid_total]
    fig = go.Figure(go.Pie(
        labels=labels, values=values,
        marker_colors=['#2ecc71', '#e74c3c'],
        textinfo='label+percent+value'
    ))
    fig.update_layout(title='Distribución de mensajes: válidos vs DLQ', height=350)
    fig.show()

In [ ]:
# ── 2.3 Consumir mensajes del DLQ y analizar razones de rechazo ───────────
import collections

N_MENSAJES_DLQ = min(invalid_total, 200)

dlq_records = []
if N_MENSAJES_DLQ > 0:
    consumer = Consumer({
        'bootstrap.servers': KAFKA_BROKER,
        'group.id': 'notebook-dlq-reader-03',
        'auto.offset.reset': 'earliest',
        'enable.auto.commit': False,
    })
    consumer.assign([TopicPartition('sensors_invalid', 0, 0)])
    leidos = 0
    while leidos < N_MENSAJES_DLQ:
        msg = consumer.poll(timeout=3.0)
        if msg is None:
            break
        if msg.error():
            continue
        try:
            data = json.loads(msg.value().decode('utf-8'))
            dlq_records.append({
                'device_id': data.get('device_id', 'unknown'),
                'reason': data.get('reason', data.get('error', 'unknown')),
                'temperature': data.get('temperature'),
                'unit': data.get('unit', ''),
            })
            leidos += 1
        except Exception:
            pass
    consumer.close()

if dlq_records:
    df_dlq = pd.DataFrame(dlq_records)
    print(f'Mensajes DLQ analizados: {len(df_dlq)}')
    print()
    reason_counts = df_dlq['reason'].value_counts()
    print('Razones de rechazo:')
    display(reason_counts.to_frame('count'))

    fig = px.bar(
        reason_counts.reset_index().rename(columns={'index': 'reason', 'reason': 'count', 'count': 'reason'}),
        x='reason', y='count',
        title=f'Causas de rechazo DLQ (muestra de {len(df_dlq)} mensajes)',
        labels={'reason': 'Motivo', 'count': 'Mensajes'}
    )
    fig.update_xaxes(tickangle=15)
    fig.update_layout(height=380)
    fig.show()
else:
    print('Sin mensajes en sensors_invalid.')

In [ ]:
# ── 2.4 Consumir muestra de sensors_clean y ver schema ────────────────────
N_SAMPLE = 50
sample_clean = []

consumer = Consumer({
    'bootstrap.servers': KAFKA_BROKER,
    'group.id': 'notebook-clean-sample-03',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
})
consumer.assign([TopicPartition('sensors_clean', 0, 0)])
leidos = 0
while leidos < N_SAMPLE:
    msg = consumer.poll(timeout=3.0)
    if msg is None:
        break
    if not msg.error():
        try:
            sample_clean.append(json.loads(msg.value().decode('utf-8')))
            leidos += 1
        except Exception:
            pass
consumer.close()

if sample_clean:
    df_sample = pd.DataFrame(sample_clean)
    print(f'Muestra sensors_clean ({len(df_sample)} mensajes):')
    print(f'Columnas: {list(df_sample.columns)}')
    print()
    print('Distribución de unidades originales:')
    display(df_sample['unit_original'].value_counts())
    print()
    print('Temperatura media por máquina (muestra Kafka):')
    display(df_sample.groupby('device_id')['temperature_c'].describe().round(2))
else:
    print('Sin mensajes en sensors_clean.')

---
## 3 — MinIO: Cold Path NDJSON

In [ ]:
# ── 3.1 Inventario de archivos en MinIO ───────────────────────────────────
mc = Minio(MINIO_ENDPOINT, access_key=MINIO_KEY, secret_key=MINIO_SECRET, secure=False)

objects = list(mc.list_objects(MINIO_BUCKET, prefix='clean/', recursive=True))
json_files = [o for o in objects if o.object_name.endswith('.json')]

print(f'Total archivos NDJSON en datalake/clean/: {len(json_files)}')

if json_files:
    file_info = [{
        'path': o.object_name,
        'size_kb': round((o.size or 0) / 1024, 1),
        'modified': o.last_modified,
    } for o in json_files]
    df_files = pd.DataFrame(file_info)
    df_files['modified'] = pd.to_datetime(df_files['modified'], utc=True)

    # Extraer particiones del path
    for col in ['year', 'month', 'day', 'hour']:
        df_files[col] = df_files['path'].str.extract(f'{col}=(\\w+)')

    print(f'Tamaño total    : {df_files["size_kb"].sum():.1f} KB')
    print(f'Archivo más nuevo: {df_files["modified"].max()}')
    print(f'Archivo más antiguo: {df_files["modified"].min()}')
    print()
    print('Últimos 10 archivos:')
    display(df_files.sort_values('modified', ascending=False).head(10)[['path', 'size_kb', 'modified']])

In [ ]:
# ── 3.2 Archivos por hora (actividad cold path) ───────────────────────────
if json_files and len(df_files) > 0:
    files_per_hour = df_files.groupby(['year', 'month', 'day', 'hour']).agg(
        archivos  = ('path', 'count'),
        total_kb  = ('size_kb', 'sum')
    ).reset_index()
    print('Archivos NDJSON por partición (año/mes/día/hora):')
    display(files_per_hour)
else:
    print('Sin archivos en MinIO. El cold path escribe cada 60s — espera 1 minuto con el pipeline corriendo.')

In [ ]:
# ── 3.3 DuckDB: leer cold path desde MinIO ────────────────────────────────
conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute(f"""
    SET s3_endpoint='{MINIO_ENDPOINT}';
    SET s3_access_key_id='{MINIO_KEY}';
    SET s3_secret_access_key='{MINIO_SECRET}';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

try:
    df_cold = conn.execute("""
        SELECT *
        FROM read_json(
            's3://datalake/clean/**/*.json',
            hive_partitioning = true,
            columns = {
                device_id:    'VARCHAR',
                temperature_c: 'DOUBLE',
                unit_original: 'VARCHAR',
                ts:            'VARCHAR',
                _ingested_at:  'BIGINT'
            }
        )
        WHERE temperature_c IS NOT NULL
        ORDER BY ts DESC
        LIMIT 5000
    """).df()

    print(f'Registros cold path cargados: {len(df_cold)}')
    if not df_cold.empty:
        print(f'Máquinas: {sorted(df_cold["device_id"].unique())}')
        print()
        display(df_cold.head(5))
except Exception as e:
    print(f'Sin datos en MinIO o error de conexión: {e}')
    df_cold = pd.DataFrame()

In [ ]:
# ── 3.4 Estadísticas cold path ────────────────────────────────────────────
if not df_cold.empty:
    stats_cold = df_cold.groupby('device_id').agg(
        registros   = ('temperature_c', 'count'),
        temp_media  = ('temperature_c', 'mean'),
        temp_max    = ('temperature_c', 'max'),
        temp_min    = ('temperature_c', 'min'),
        desv_std    = ('temperature_c', 'std'),
    ).round(2)

    unidades = df_cold.groupby('unit_original').size().rename('registros')

    print('Estadísticas por máquina (cold path MinIO):')
    display(stats_cold)
    print()
    print('Distribución de unidades de origen:')
    display(unidades.to_frame())

    fig = px.pie(
        unidades.reset_index().rename(columns={'index': 'unit_original'}),
        names='unit_original', values='registros',
        title='Proporción de unidades en cold path (C / F / K)',
        color_discrete_sequence=['#3498db','#e67e22','#2ecc71']
    )
    fig.show()
else:
    print('Cold path vacío — sin datos que analizar.')

---
## 4 — FastAPI: Capa de servicio

In [ ]:
# ── 4.1 Helper para llamar a la API ───────────────────────────────────────
def api_get(path, timeout=8):
    try:
        url = f'{API_URL}{path}'
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return json.loads(r.read()), None
    except Exception as e:
        return None, str(e)

def api_post(path, body=None, timeout=15):
    try:
        url = f'{API_URL}{path}'
        data = json.dumps(body).encode() if body else b''
        req = urllib.request.Request(url, data=data, method='POST',
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read()), None
    except Exception as e:
        return None, str(e)

# Health check
health, err = api_get('/health')
if health:
    print('FastAPI /health:')
    for k, v in health.items():
        print(f'  {k}: {v}')
else:
    print(f'FastAPI no disponible: {err}')

In [ ]:
# ── 4.2 Estado de todas las máquinas ─────────────────────────────────────
machines, err = api_get('/machines/status')
if machines:
    df_api = pd.DataFrame(machines.get('machines', []))
    if not df_api.empty:
        print(f'Máquinas con datos: {len(df_api)}')
        display(df_api)

        # Gráfica de gauges por máquina
        fig = make_subplots(
            rows=1, cols=len(df_api),
            specs=[[{"type": "indicator"}] * len(df_api)]
        )
        for i, row in enumerate(df_api.itertuples(), 1):
            temp = getattr(row, 'last_temp_c', getattr(row, 'avg_temp_c', 0)) or 0
            color = 'red' if temp > 80 else ('orange' if temp > 70 else 'green')
            fig.add_trace(go.Indicator(
                mode='gauge+number',
                value=temp,
                title={'text': row.device_id, 'font': {'size': 12}},
                gauge={
                    'axis': {'range': [0, 120]},
                    'bar': {'color': color},
                    'threshold': {'line': {'color': 'red', 'width': 3}, 'value': 80}
                }
            ), row=1, col=i)
        fig.update_layout(title='Temperatura actual por máquina (FastAPI)', height=300)
        fig.show()
    else:
        print('Sin máquinas con datos (InfluxDB vacío o pipeline no iniciado).')
else:
    print(f'Error: {err}')

In [ ]:
# ── 4.3 Alertas activas ───────────────────────────────────────────────────
alerts_api, err = api_get('/alerts')
if alerts_api:
    alert_list = alerts_api.get('alerts', [])
    print(f'Ventanas en alerta: {len(alert_list)}')
    if alert_list:
        df_alert_api = pd.DataFrame(alert_list)
        display(df_alert_api.head(20))
else:
    print(f'Error en /alerts: {err}')

In [ ]:
# ── 4.4 Entrenar modelo IA y hacer predicciones en rango de temperaturas ──
model_status, _ = api_get('/model/status')
print('Estado modelo IA:')
print(json.dumps(model_status, indent=2))

# Entrenar si no está entrenado
if model_status and not model_status.get('trained', False):
    print('\nEntrenando modelo...')
    train_result, err = api_post('/model/train?range_minutes=60&contamination=0.1')
    if train_result:
        print(json.dumps(train_result, indent=2))
    else:
        print(f'Error entrenando: {err}')

In [ ]:
# ── 4.5 Predicciones IA para un rango de temperaturas ────────────────────
model_status, _ = api_get('/model/status')
if model_status and model_status.get('trained'):
    device = 'machine-004'  # máquina más caliente
    temps = list(range(40, 105, 5))
    preds = []
    for t in temps:
        result, _ = api_get(f'/machines/{device}/predict?temperature_c={t}')
        if result:
            preds.append({
                'temperature_c': t,
                'is_anomaly': result.get('is_anomaly', False),
                'failure_prob': result.get('failure_prob', 0),
            })

    df_pred = pd.DataFrame(preds)
    display(df_pred)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_pred['temperature_c'], y=df_pred['failure_prob'],
        mode='lines+markers',
        marker=dict(
            color=df_pred['failure_prob'],
            colorscale='RdYlGn_r',
            size=10
        ),
        name='failure_prob'
    ))
    fig.add_vline(x=80, line_dash='dash', line_color='red', annotation_text='Umbral Flink 80°C')
    fig.add_hline(y=0.5, line_dash='dot', line_color='orange', annotation_text='Umbral ML 0.5')
    fig.update_layout(
        title=f'Probabilidad de fallo IsolationForest — {device}',
        xaxis_title='Temperatura (°C)', yaxis_title='failure_prob',
        yaxis_range=[0, 1.1], height=400
    )
    fig.show()
else:
    print('Modelo no entrenado. Ejecuta la celda anterior primero.')

---
## 5 — Lambda Query: Hot + Cold unificados

In [ ]:
# ── 5.1 Preparar hot path DataFrame ──────────────────────────────────────
hot_available = not df_hot.empty
cold_available = not df_cold.empty

print(f'Hot path  (InfluxDB): {"disponible" if hot_available else "sin datos"}')
print(f'Cold path (MinIO)   : {"disponible" if cold_available else "sin datos"}')

if hot_available:
    hot_lambda = df_hot[['_time', 'device_id', 'avg_temp_c']].rename(
        columns={'_time': 'ts', 'avg_temp_c': 'temperature_c'}
    ).copy()
    hot_lambda['source'] = 'InfluxDB (hot)'
    hot_lambda['ts'] = hot_lambda['ts'].astype(str)
    print(f'  {len(hot_lambda)} filas en hot_lambda')

if cold_available:
    cold_lambda = df_cold[['ts', 'device_id', 'temperature_c']].copy()
    cold_lambda['source'] = 'MinIO (cold)'
    print(f'  {len(cold_lambda)} filas en cold_lambda')

In [ ]:
# ── 5.2 UNION ALL con DuckDB ──────────────────────────────────────────────
if hot_available and cold_available:
    conn2 = duckdb.connect()
    conn2.register('hot',  hot_lambda)
    conn2.register('cold', cold_lambda)

    unified = conn2.execute("""
        SELECT device_id, CAST(temperature_c AS DOUBLE) AS temperature_c, ts, source
        FROM hot
        UNION ALL
        SELECT device_id, CAST(temperature_c AS DOUBLE) AS temperature_c, ts, source
        FROM cold
        ORDER BY ts DESC
        LIMIT 5000
    """).df()

    print(f'Total registros unificados: {len(unified)}')
    print(unified.groupby('source').agg(registros=('temperature_c','count'), temp_media=('temperature_c','mean')).round(2))

    fig = px.scatter(
        unified, x='ts', y='temperature_c',
        color='source', symbol='device_id',
        title='Lambda Query: Hot + Cold unificados (DuckDB UNION ALL)',
        labels={'ts': 'Tiempo', 'temperature_c': '°C', 'source': 'Fuente'},
        opacity=0.7, height=450
    )
    fig.add_hline(y=80, line_dash='dash', line_color='red')
    fig.show()
elif hot_available:
    print('Solo hot path disponible (cold path sin datos aún).')
    print('El cold path escribe cada 60s — espera 1 minuto con kafka_to_minio.py corriendo.')
else:
    print('Sin datos suficientes para la query federada.')

In [ ]:
# ── 5.3 Cobertura temporal: ¿hot y cold se solapan? ──────────────────────
if hot_available and cold_available:
    hot_range  = (hot_lambda['ts'].min(), hot_lambda['ts'].max())
    cold_range = (cold_lambda['ts'].min(), cold_lambda['ts'].max())

    print('Cobertura temporal:')
    print(f'  Hot  (InfluxDB): {hot_range[0]}  →  {hot_range[1]}')
    print(f'  Cold (MinIO)   : {cold_range[0]}  →  {cold_range[1]}')

    # Registros por máquina en cada source
    coverage = unified.groupby(['device_id', 'source']).size().unstack(fill_value=0)
    print()
    print('Registros por máquina y fuente:')
    display(coverage)

---
## 6 — Cálculos Estadísticos Avanzados

In [ ]:
# ── 6.1 Z-score por máquina: identificar outliers ─────────────────────────
if not df_hot.empty:
    df_zscore = df_hot.copy()
    df_zscore['z_score'] = df_zscore.groupby('device_id')['avg_temp_c'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df_zscore['outlier'] = df_zscore['z_score'].abs() > 2.0

    n_outliers = df_zscore['outlier'].sum()
    print(f'Outliers detectados (|z| > 2): {n_outliers} de {len(df_zscore)} ventanas ({n_outliers/len(df_zscore)*100:.1f}%)')
    print()

    fig = px.scatter(
        df_zscore, x='_time', y='z_score', color='device_id',
        symbol=df_zscore['outlier'].map({True: 'x', False: 'circle'}),
        title='Z-score de temperatura por máquina (x = outlier |z|>2)',
        labels={'_time': 'Tiempo', 'z_score': 'Z-score', 'device_id': 'Máquina'}
    )
    fig.add_hline(y=2,  line_dash='dash', line_color='orange')
    fig.add_hline(y=-2, line_dash='dash', line_color='orange')
    fig.add_hline(y=0,  line_color='gray', line_width=1)
    fig.update_layout(height=400)
    fig.show()
else:
    print('Sin datos InfluxDB para calcular Z-score.')

In [ ]:
# ── 6.2 Media móvil y banda de Bollinger (±2σ) ───────────────────────────
if not df_hot.empty:
    WINDOW = 5  # ventanas de 1 min → media móvil de 5 min
    device_focus = df_hot.groupby('device_id')['avg_temp_c'].mean().idxmax()  # la más caliente
    d = df_hot[df_hot['device_id'] == device_focus].sort_values('_time').copy()

    d['rolling_mean'] = d['avg_temp_c'].rolling(WINDOW, min_periods=1).mean()
    d['rolling_std']  = d['avg_temp_c'].rolling(WINDOW, min_periods=1).std().fillna(0)
    d['upper_band']   = d['rolling_mean'] + 2 * d['rolling_std']
    d['lower_band']   = d['rolling_mean'] - 2 * d['rolling_std']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d['_time'], y=d['upper_band'], fill=None, mode='lines',
                             line_color='rgba(200,0,0,0.2)', name='Banda sup (+2σ)'))
    fig.add_trace(go.Scatter(x=d['_time'], y=d['lower_band'], fill='tonexty', mode='lines',
                             line_color='rgba(200,0,0,0.2)', fillcolor='rgba(200,0,0,0.07)',
                             name='Banda inf (-2σ)'))
    fig.add_trace(go.Scatter(x=d['_time'], y=d['avg_temp_c'],
                             mode='lines+markers', name='avg_temp_c', line_color='steelblue'))
    fig.add_trace(go.Scatter(x=d['_time'], y=d['rolling_mean'],
                             mode='lines', name=f'Media móvil {WINDOW}min', line=dict(color='navy', dash='dash')))
    fig.add_hline(y=80, line_dash='dot', line_color='red', annotation_text='80°C')
    fig.update_layout(
        title=f'Bollinger Bands — {device_focus} (ventana {WINDOW} min)',
        xaxis_title='Tiempo', yaxis_title='°C', height=420
    )
    fig.show()

In [ ]:
# ── 6.3 Heatmap de correlación entre máquinas ─────────────────────────────
if not df_hot.empty and df_hot['device_id'].nunique() > 1:
    pivot = df_hot.pivot_table(
        index='_time', columns='device_id', values='avg_temp_c', aggfunc='mean'
    ).dropna(how='all')

    if not pivot.empty and len(pivot) > 2:
        corr = pivot.corr()
        fig = px.imshow(
            corr, text_auto='.2f',
            color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
            title='Correlación de temperatura entre máquinas (Pearson)',
        )
        fig.update_layout(height=400)
        fig.show()
        print('\nMáquinas más correlacionadas:')
        pairs = []
        for i, a in enumerate(corr.columns):
            for b in corr.columns[i+1:]:
                pairs.append({'machine_A': a, 'machine_B': b, 'corr': round(corr.loc[a, b], 3)})
        display(pd.DataFrame(pairs).sort_values('corr', ascending=False))
    else:
        print('Datos insuficientes para calcular correlación.')

In [ ]:
# ── 6.4 Tendencia lineal por máquina (¿están subiendo las temperaturas?) ──
if not df_hot.empty:
    results_trend = []
    for device in sorted(df_hot['device_id'].unique()):
        d = df_hot[df_hot['device_id'] == device].sort_values('_time').copy()
        if len(d) < 3:
            continue
        x = np.arange(len(d))
        y = d['avg_temp_c'].values
        coef = np.polyfit(x, y, 1)
        slope = coef[0]  # °C por ventana de 1 min
        # Línea de tendencia
        trend_line = np.poly1d(coef)(x)

        direction = '🔺 subiendo' if slope > 0.1 else ('🔻 bajando' if slope < -0.1 else '➡️  estable')
        results_trend.append({
            'machine': device,
            'slope_degC_per_min': round(slope, 4),
            'tendencia': direction,
            'temp_inicio': round(trend_line[0], 2),
            'temp_final':  round(trend_line[-1], 2),
        })

    df_trend = pd.DataFrame(results_trend)
    print('Tendencia de temperatura por máquina (regresión lineal):')
    display(df_trend)

    fig = px.bar(
        df_trend, x='machine', y='slope_degC_per_min',
        color='slope_degC_per_min', color_continuous_scale='RdBu_r',
        color_continuous_midpoint=0,
        title='Pendiente de temperatura (°C/min) — positivo = subiendo',
        labels={'machine': 'Máquina', 'slope_degC_per_min': '°C por min'}
    )
    fig.add_hline(y=0, line_color='black')
    fig.update_layout(height=350)
    fig.show()

In [ ]:
# ── 6.5 Resumen ejecutivo ─────────────────────────────────────────────────
print('══════════════════════════════════════════════════')
print('   RESUMEN EJECUTIVO DEL PIPELINE')
print('══════════════════════════════════════════════════')
print()

# InfluxDB
if not df_hot.empty:
    total_ventanas  = len(df_hot)
    ventanas_alerta = (df_hot['alert'] == 1).sum()
    print(f'InfluxDB (hot path):')
    print(f'  Ventanas totales     : {total_ventanas}')
    print(f'  Ventanas en alerta   : {ventanas_alerta} ({ventanas_alerta/total_ventanas*100:.1f}%)')
    print(f'  Temp. media global   : {df_hot["avg_temp_c"].mean():.2f} °C')
    print(f'  Máquina más caliente : {df_hot.groupby("device_id")["avg_temp_c"].mean().idxmax()}')
    print()

# Kafka
print(f'Redpanda (Kafka):')
for topic, wm in watermarks.items():
    print(f'  {topic:<24}: {wm["mensajes"]:>6} mensajes')
print()

# MinIO
print(f'MinIO (cold path):')
print(f'  Archivos NDJSON: {len(json_files)}')
if not df_cold.empty:
    print(f'  Registros leídos: {len(df_cold)}')
print()

# Calidad de datos
if raw_total > 0:
    print(f'Calidad de datos:')
    print(f'  Tasa de aceptación  : {clean_total/raw_total*100:.1f}%')
    print(f'  Tasa de rechazo DLQ : {invalid_total/raw_total*100:.1f}%')
print()
print('══════════════════════════════════════════════════')